In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install menpo
!git clone https://github.com/im-xiaoming/HuyenLaoNhao.git

In [ ]:
!cp -r /content/drive/MyDrive/Ying/data.zip /content/
!unzip /content/data.zip -d /content/
!cp -r /content/drive/MyDrive/Ying/mm.dat /content
!cp -r /content/drive/MyDrive/Ying/mm.dat.conf /content

In [ ]:
!cp -r /content/drive/MyDrive/Ying/faces_extracted.zip /content
!unzip -q /content/faces_extracted.zip -d /content/

!cp -r /content/drive/MyDrive/Ying/IJBB.zip /content
!unzip -q /content/IJBB.zip -d /content
!cp -r /content/drive/MyDrive/IJB_meta.zip /content
!unzip -q /content/IJB_meta.zip -d /content
!cp -r /content/meta/IJBB_meta/* /content/IJBB/meta

In [ ]:
# !cp -r /content/drive/MyDrive/ijb-testsuite.tar /content
# !file ijb-testsuite.tar
# !tar -xf ijb-testsuite.tar
# !cp -r /content/drive/MyDrive/IJB_meta.zip /content
# !unzip -q /content/IJB_meta.zip -d /content
# !cp -r /content/meta/IJBB_meta/* /content/ijb/IJBB/meta
# !cp -r /content/meta/IJBC_meta/* /content/ijb/IJBC/meta

In [ ]:
import numpy as np
from tqdm import tqdm
import os
import json
from PIL import Image
import cv2

import torch
import torchvision.transforms as transforms
import torch.nn as nn
from torch.utils.data import DataLoader

from xiaoying.validation import evaluate_utils, evaluate
from xiaoying import net
from xiaoying.data import CustomImageFolderDataset, val_dataset
from xiaoying.head import AdaFace, SubAdaFace
from xiaoying.utils import EarlyStopping, load_checkpoint, load_weights, split_parameters

In [ ]:
root = '/content/faces_extracted'
print("Num claass: ", len(os.listdir(root)))

train_transforms = transforms.Compose([
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
    ])

train_dataset = CustomImageFolderDataset('/content/faces_extracted',
                                         transform=train_transforms)
train_loader = DataLoader(
    train_dataset,
    batch_size=256,
    shuffle=True,
    num_workers=8,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=4,
    drop_last=True
)


val_ds = val_dataset(data_root='/content/data')
val_loader = DataLoader(val_ds, batch_size=512, shuffle=False,
                        num_workers=4, pin_memory=True)

In [ ]:
model_name = 'ir_18'
model = net.build_model(model_name)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

criterion = nn.CrossEntropyLoss()
head = AdaFace(embedding_size=512, classnum=8631, m=0.4, h=0.333, s=64., t_alpha=0.99)
head.to(device)

paras_wo_bn, paras_only_bn = split_parameters(model)

optimizer = torch.optim.SGD([{
            'params': paras_wo_bn + [head.kernel],
            'weight_decay': 1e-4
        }, {
            'params': paras_only_bn
        }], lr=0.01, momentum=0.9)

In [ ]:
def validate(model, dataname, val_loader, device):
    # Evaluate
    print('Evaluate step 1...')
    mean_acc = evaluate.evaluate1(model, val_loader, device)

    print('Evaluate step 2...')
    r = evaluate.evaluate2('', model, 'IR', 'IJBB', 256, device=device)

    return mean_acc, r

In [ ]:
def train(model, head, train_loader, optimizer,
          criterion, epochs, device, **kwargs):
    early_stopping = EarlyStopping('/content', 'checkpoint_backups/',
                               '/content/drive/MyDrive/checkpoints',
                                   model, head, optimizer)

    # load checkpoint
    start = 0 if not kwargs.get('filename') else \
                    load_checkpoint(kwargs.get('filename'), model, head, optimizer,
                                    early_stopping, device)
    optimizer = torch.optim.SGD([{
            'params': paras_wo_bn + [head.kernel],
            'weight_decay': 1e-4
        }, {
            'params': paras_only_bn
        }], lr=0.0001, momentum=0.9)

    for epoch in range(start, epochs):
        # ==================== Training ==================== #
        model.train()

        train_loss = []
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")
        print('\nTrain...')
        for images, labels in pbar:

            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()

            # forward
            embedings, norm = model(images)
            cos_theta = head(embedings, norm, labels)
            loss = criterion(cos_theta, labels)

            # backward
            loss.backward()
            optimizer.step()

            train_loss.append(loss.item())

            # update tqdm
            pbar.set_postfix({
                "loss": f"{np.mean(train_loss):.4f}",
                "lr": optimizer.param_groups[0]["lr"]
            })

        early_stopping._save() # save backup

        # ==================== Validation ==================== #
        model.eval()
        mean_acc, r = validate(model, kwargs.get('dataname'), val_loader, device)

        early_stopping(**{
            'acc': r['0.0001'],
            'epoch': epoch+1,
            'fpr_1e-4': r['0.0001'],
            'fpr_1e-5': r['1e-05'],
        })
        print(f'Loss: {np.mean(train_loss):.4f} - Acc: {mean_acc:.4f}\n')

        if early_stopping.stop:
            print("Early Stopping")
            break

In [ ]:
args = {
    'root': '/content',
    'model_name': model_name,
    'filename': '/content/drive/MyDrive/checkpoints/checkpoint_5.pth',
    'dataname': 'IJBB'
}

train(model, head, train_loader, optimizer, criterion, epochs=7, device=device, **args)